# Table S3 — Candidate Hub Regulators by Outgoing Regulatory Strength

This notebook generates **Table S3**: ranked lists of candidate hub regulators for each experimental dataset (Semrau, Kameneva, Schiebinger), based on **outgoing regulatory strength** in the GRN inferred by CardamomOT.

**Definition**: Outgoing regulatory strength = $\sum_{j \neq i} |W_{ij}|$, i.e. the sum of absolute values of all outgoing edge weights from gene $i$, excluding self-loops.

In [1]:
import numpy as np
import scanpy as sc
import pandas as pd
import os
import sys; sys.path += ['./../../']

In [2]:
def build_hub_table(datapath, dataset_name, stimulus_names):
    """
    Build a ranked hub regulator table from inferred GRN.

    Parameters
    ----------
    datapath : str
        Path to the experimental dataset folder.
    dataset_name : str
        Name of the dataset (e.g. 'Semrau').
    stimulus_names : list of str
        Names of stimulus genes (e.g. ['RA'] or ['Dox', 'Serum']).

    Returns
    -------
    pd.DataFrame with columns:
        Rank, Dataset, Gene, Type, Outgoing_Strength,
        N_Outgoing_Edges, Top5_Outgoing_Targets
    """
    # --- Load gene names ---
    full_path = os.path.join(datapath, 'Data', 'data_full.h5ad')
    train_path = os.path.join(datapath, 'Data', 'data_train.h5ad')
    adata = sc.read_h5ad(full_path if os.path.exists(full_path) else train_path)
    gene_names = list(stimulus_names) + list(adata.var_names)

    # --- Load GRN matrix ---
    grn_path = os.path.join(datapath, 'cardamomOT', 'inter_simul.npy')
    grn_mat = np.load(grn_path)
    matrix = grn_mat[:, :, 0] if grn_mat.ndim == 3 else grn_mat

    n_genes = len(gene_names)
    n_stim = len(stimulus_names)

    # --- Remove self-loops ---
    off_diag = matrix.copy()
    np.fill_diagonal(off_diag, 0)

    # --- Outgoing regulatory strength ---
    out_strength = np.abs(off_diag).sum(axis=1)
    n_out_edges = (off_diag != 0).sum(axis=1)

    # --- Build table ---
    rows = []
    for i in range(n_genes):
        name = gene_names[i]
        gene_type = 'Stimulus' if i < n_stim else 'Gene'
        strength = out_strength[i]
        n_edges = n_out_edges[i]

        # Top 5 outgoing targets with signed weights
        out_weights = off_diag[i, :]
        top5_idx = np.argsort(np.abs(out_weights))[::-1][:5]
        top5_parts = []
        for j in top5_idx:
            w = out_weights[j]
            if w != 0:
                top5_parts.append(f'{gene_names[j]} ({w:+.2f})')
        top5_targets = '; '.join(top5_parts) if top5_parts else '—'

        rows.append({
            'Dataset': dataset_name,
            'Gene': name,
            'Type': gene_type,
            'Outgoing_Strength': round(strength, 3),
            'N_Outgoing_Edges': n_edges,
            'Top5_Outgoing_Targets': top5_targets,
        })

    df = pd.DataFrame(rows)
    df = df.sort_values('Outgoing_Strength', ascending=False)
    df['Rank'] = range(1, len(df) + 1)
    df = df[['Rank', 'Dataset', 'Gene', 'Type', 'Outgoing_Strength',
             'N_Outgoing_Edges', 'Top5_Outgoing_Targets']]
    return df

In [3]:
# --- Build hub tables for all three datasets ---
df_semrau = build_hub_table('./../../experimental_datasets/Semrau', 'Semrau', ['RA'])
df_kameneva = build_hub_table('./../../experimental_datasets/Kameneva', 'Kameneva', ['RA'])
df_schiebinger = build_hub_table('./../../experimental_datasets/Schiebinger', 'Schiebinger', ['Dox', 'Serum'])

# Combine into a single table
df_all = pd.concat([df_semrau, df_kameneva, df_schiebinger], ignore_index=True)

# Save to CSV
csv_path = 'tableS3_hub_regulators.csv'
df_all.to_csv(csv_path, index=False)
print(f'Saved {len(df_all)} rows to {csv_path}')
print(f'  Semrau:    {len(df_semrau)} genes')
print(f'  Kameneva:  {len(df_kameneva)} genes')
print(f'  Schiebinger: {len(df_schiebinger)} genes')

Saved 250 rows to tableS3_hub_regulators.csv
  Semrau:    42 genes
  Kameneva:  98 genes
  Schiebinger: 110 genes


## Semrau — Top 15 hub regulators (genes only, excluding stimuli)

In [4]:
def display_top_hubs(df, dataset_name, top_n=15):
    """Display top N hub genes for a dataset, excluding stimuli."""
    subset = df[(df['Dataset'] == dataset_name) & (df['Type'] == 'Gene')].head(top_n)
    return subset[['Rank', 'Gene', 'Outgoing_Strength', 'N_Outgoing_Edges', 'Top5_Outgoing_Targets']]

display_top_hubs(df_all, 'Semrau')

,Rank,Gene,Outgoing_Strength,N_Outgoing_Edges,Top5_Outgoing_Targets
0,1,Hoxb2,158.362,17,Zfp42 (-42.34); Pou5f1 (-28.58); Cd24a (+20.41...
1,2,Lamb1,70.049,12,Sparc (+21.56); Col4a2 (+15.11); Gata6 (+7.80)...
3,4,Dnmt3a,59.864,23,Nr0b1 (-6.54); Esrrb (-5.37); Pou5f1 (-4.69); ...
4,5,Col4a2,47.412,10,Lamb1 (+17.72); Sparc (+16.99); Gata4 (+6.88);...
5,6,Klf2,27.232,16,Dppa2 (+13.97); Zic2 (+2.38); Zfp42 (+2.03); K...
6,7,Esrrb,25.699,22,Dppa4 (+3.89); Klf4 (+3.09); Nr0b1 (+2.60); Dn...
7,8,Sox2,25.067,11,Pou5f1 (+4.37); Esrrb (+3.52); Zic3 (+3.46); J...
8,9,Dppa2,22.443,16,Hoxa1 (+3.15); Hoxb2 (+2.98); Klf4 (-2.73); Kl...
9,10,Sparc,17.697,13,Klf4 (+5.34); Lamb1 (+3.95); Krt8 (+2.07); Pdg...
10,11,Klf4,15.902,14,Dppa4 (+2.75); Klf2 (+2.46); Pou5f1 (+2.11); Z...


## Kameneva — Top 15 hub regulators

In [5]:
display_top_hubs(df_all, 'Kameneva')

,Rank,Gene,Outgoing_Strength,N_Outgoing_Edges,Top5_Outgoing_Targets
42,1,RPL30,162.587,82,RPL34 (+8.31); RPL10 (+7.40); RPL32 (+6.76); R...
43,2,MEG3,146.010,76,RPS2 (+4.94); RPL41 (+4.30); COL18A1 (+3.98); ...
44,3,HAND2,106.099,50,STMN2 (+7.12); PHOX2A (+6.12); PTPRZ1 (-5.79);...
45,4,CHGA,78.626,38,RAMP1 (+7.20); PENK (+5.44); CARTPT (+4.21); P...
47,6,ALDH1A1,59.370,83,RPL10 (-2.49); RPS29 (-2.18); FAU (-2.11); RPL...
48,7,POSTN,56.965,80,COL5A2 (+2.31); RPL34 (-1.87); RPL12 (-1.60); ...
49,8,HTATSF1,49.207,47,PENK (+2.53); RPL41 (+2.29); HINT1 (+2.22); PN...
50,9,ZEB2,46.498,66,RPS2 (-2.04); PNMT (-1.97); RPL32 (-1.83); PCS...
51,10,PCSK1N,46.469,41,ATP5F1E (+5.21); FAU (+4.55); HINT1 (+4.08); U...
52,11,HAND2-AS1,40.594,50,MAP1B (+2.78); BASP1 (+2.44); CD24 (+2.36); EE...


## Schiebinger — Top 15 hub regulators

In [6]:
display_top_hubs(df_all, 'Schiebinger')

,Rank,Gene,Outgoing_Strength,N_Outgoing_Edges,Top5_Outgoing_Targets
140,1,Fabp3,73.662,101,Tcl1 (+5.68); Mybl2 (+3.21); Dppa5a (+3.11); M...
142,3,Dppa5a,46.461,104,Tcl1 (+6.33); Cdkn2a (-5.97); Lncenc1 (+3.57);...
144,5,Ereg,26.735,104,Fam20c (+1.81); Myc (+1.78); Ddit3 (-1.66); Id...
145,6,Obox6,23.778,104,Spic (+4.18); Dppa4 (+3.85); Hesx1 (+3.30); Ln...
146,7,Txnip,23.507,104,Hist1h2ap (-1.36); Nme2 (-1.24); 2810417H13Rik...
147,8,Dmrtc2,22.249,101,Tmem35 (-1.79); Epcam (+1.19); Id3 (-1.16); Cd...
148,9,Cdkn2a,22.151,101,2810417H13Rik (-0.97); Tax1bp3 (+0.80); Gabara...
149,10,S100a6,22.091,103,Dppa5a (-3.47); Col1a2 (+0.73); Prss23 (+0.73)...
150,11,Tnfrsf12a,21.375,103,Ereg (+1.72); Myc (+0.79); 2810417H13Rik (+0.7...
151,12,Gm26917,20.426,104,Vps8 (+1.32); Vbp1 (+1.24); Erdr1 (+1.17); Sca...


## Stimulus regulatory strength (for reference)

Shows how strongly each stimulus gene regulates the GRN.

In [7]:
df_all[df_all['Type'] == 'Stimulus'][['Dataset', 'Gene', 'Outgoing_Strength', 'N_Outgoing_Edges', 'Top5_Outgoing_Targets']]

,Dataset,Gene,Outgoing_Strength,N_Outgoing_Edges,Top5_Outgoing_Targets
2,Semrau,RA,63.911,30,Klf2 (-5.85); Dppa2 (+5.27); Esrrb (-5.00); Fg...
46,Kameneva,RA,74.655,55,S100B (-3.81); OLFML2A (-3.60); PLP1 (-3.53); ...
141,Schiebinger,Dox,50.610,102,Dppa5a (-3.52); Dmrtc2 (+3.38); Tmem35 (+2.28)...
143,Schiebinger,Serum,28.483,105,Tnfrsf12a (-1.98); Gadd45b (-1.94); Lox (-1.79...


## LaTeX Table — Top 10 Hub Regulators per Dataset (Vertical Layout)

LaTeX-formatted table spanning the full page width for inclusion in the manuscript. Datasets are stacked vertically in a single table using `tabularx` with `booktabs` formatting.

In [8]:
def escape_latex(s):
    """Escape special LaTeX characters in a string."""
    return (str(s)
            .replace('\\', r'\textbackslash ')
            .replace('_', r'\_')
            .replace('&', r'\&')
            .replace('%', r'\%')
            .replace('#', r'\#')
            .replace('$', r'\$')
            .replace('{', r'\{')
            .replace('}', r'\}')
            .replace('~', r'\textasciitilde ')
            .replace('^', r'\textasciicircum '))


def _top3_targets_no_weights(targets_str):
    """Extract top 3 target gene names (without interaction weights) from the targets string."""
    import re
    parts = [p.strip() for p in targets_str.split(';')]
    gene_names = []
    for p in parts:
        # Strip the weight in parentheses, e.g., "Zfp42 (-28.19)" -> "Zfp42"
        name = re.sub(r'\s*\([+-]?[\d.]+\)\s*$', '', p).strip()
        if name and name != '—':
            gene_names.append(escape_latex(name))
    return ', '.join(gene_names[:3]) if gene_names else '—'


def generate_latex_vertical_table(df_all, top_n=10):
    """
    Generate a single vertical LaTeX table spanning the full page width.
    Datasets are stacked vertically with grouping headers.
    Uses tabularx with booktabs for a clean, compact layout.
    """
    newline = '\n'
    lines = []

    lines.append(r'\begin{table}[ht]')
    lines.append(r'\centering')
    lines.append(r'\caption{Top \detokenize{10} candidate hub regulators ranked by '
                  r'outgoing regulatory strength $\sum_{j\neq i}|W_{ij}|$ '
                  r'in the GRN inferred by \textsc{CardamomOT} for each '
                  r'experimental dataset.}')
    lines.append(r'\label{tableS3}')
    lines.append(r'\small')
    lines.append(r'\begin{tabularx}{\textwidth}{r l l l r X}')
    lines.append(r'\toprule')
    lines.append(r'Dataset & Rank & Gene & Out.\ Strength & $N$ Edges & Top 3 Outgoing Targets \\')
    lines.append(r'\midrule')

    datasets = ['Semrau', 'Kameneva', 'Schiebinger']
    for ds_idx, ds_name in enumerate(datasets):
        sub = df_all[(df_all['Dataset'] == ds_name) & (df_all['Type'] == 'Gene')].head(top_n)

        for row_idx, (_, row) in enumerate(sub.iterrows()):
            gene = escape_latex(row['Gene'])
            strength = f"{row['Outgoing_Strength']:.3f}"
            n_edges = row['N_Outgoing_Edges']
            targets = _top3_targets_no_weights(row['Top5_Outgoing_Targets'])

            # Show dataset name only on first row of each group
            ds_cell = ds_name if row_idx == 0 else ''
            rank = row_idx + 1

            lines.append(f'{ds_cell} & {rank} & {gene} & {strength} & {n_edges} & {targets} \\\\')

        # Add a separating rule between datasets (but not after the last)
        if ds_idx < len(datasets) - 1:
            lines.append(r'\cmidrule{1-6}')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabularx}')
    lines.append(r'\end{table}')

    return newline.join(lines)


latex_output = generate_latex_vertical_table(df_all, top_n=10)
print(latex_output)

# Also save to a .tex file for direct inclusion
with open('tableS3_hub_regulators.tex', 'w') as f:
    f.write(latex_output)
print('\nSaved to tableS3_hub_regulators.tex')

\begin{table}[ht]
\centering
\caption{Top \detokenize{10} candidate hub regulators ranked by outgoing regulatory strength $\sum_{j\neq i}|W_{ij}|$ in the GRN inferred by \textsc{CardamomOT} for each experimental dataset.}
\label{tableS3}
\small
\begin{tabularx}{\textwidth}{r l l l r X}
\toprule
Dataset & Rank & Gene & Out.\ Strength & $N$ Edges & Top 3 Outgoing Targets \\
\midrule
Semrau & 1 & Hoxb2 & 158.362 & 17 & Zfp42, Pou5f1, Cd24a \\
 & 2 & Lamb1 & 70.049 & 12 & Sparc, Col4a2, Gata6 \\
 & 3 & Dnmt3a & 59.864 & 23 & Nr0b1, Esrrb, Pou5f1 \\
 & 4 & Col4a2 & 47.412 & 10 & Lamb1, Sparc, Gata4 \\
 & 5 & Klf2 & 27.232 & 16 & Dppa2, Zic2, Zfp42 \\
 & 6 & Esrrb & 25.699 & 22 & Dppa4, Klf4, Nr0b1 \\
 & 7 & Sox2 & 25.067 & 11 & Pou5f1, Esrrb, Zic3 \\
 & 8 & Dppa2 & 22.443 & 16 & Hoxa1, Hoxb2, Klf4 \\
 & 9 & Sparc & 17.697 & 13 & Klf4, Lamb1, Krt8 \\
 & 10 & Klf4 & 15.902 & 14 & Dppa4, Klf2, Pou5f1 \\
\cmidrule{1-6}
Kameneva & 1 & RPL30 & 162.587 & 82 & RPL34, RPL10, RPL32 \\
 & 2 & MEG3 & 1